# Phase 8 — Stress Testing & Impact Analysis Validation Notebook

This notebook validates the **production** `backend/simulation/stress.py` and
`backend/simulation/sensitivity.py` modules against the frozen
`docs/01_architecture/simulation.md` §5 outputs and §6 invariants: the
baseline-vs-stressed comparison, KPI deltas, the breach policy, the monthly
EBITDA waterfall with its below-EBITDA bridge to net income, and single-factor
sweeps.

Everything is deterministic and local: no network, no API keys, no LLM calls,
no database. Inputs are the synthetic fixture companies used by the test suite.

In [1]:
import sys
from pathlib import Path

# Path bootstrap so a fresh kernel can import the repository package.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from backend.data_engine.contracts import SyntheticCompanyConfig
from backend.data_engine.ingest.synthetic import generate_company
from backend.simulation import scenarios as sc
from backend.simulation import sensitivity as sn
from backend.simulation import stress as st
from backend.simulation.contracts import REGISTRY_VERSION, TWIN_VERSION

print("stress version:  ", st.STRESS_VERSION)
print("scenario version:", sc.SCENARIO_VERSION)
print("twin version:    ", TWIN_VERSION)
print("registry version:", REGISTRY_VERSION)
print("sweepable fields:", len(sn.SWEEPABLE_FIELDS), "| grid points:", sn.SWEEP_POINTS)

stress version:   1.0.0
scenario version: 1.0.0
twin version:     1.0.0
registry version: 1.0.0
sweepable fields: 9 | grid points: 9


## 1. Baseline vs stressed on the same deterministic twin path

`run_stress()` runs `simulate()` twice — once with zero overrides (baseline)
and once with the validated scenario's overrides — then compares. Seed 1001
under the frozen `recession` preset (−15% revenue, +10 AR days).

In [2]:
dataset = generate_company(SyntheticCompanyConfig(seed=1001, periods=24))
recession = sc.get_preset("recession")
result = st.run_stress(dataset, recession)

print("scenario:", result.scenario_name, "| horizon:", result.horizon_months)
print("result hash:", result.result_hash[:16], "...")
print("dimension deltas:", result.dimension_deltas, "->", result.dimension_deltas_note["status"])

for metric in ("revenue", "ebitda", "net_income", "min_cash"):
    trough = next(
        i for i in result.kpi_impacts if i.metric == metric and i.scope == "trough"
    )
    pct = "n/a" if trough.delta_pct is None else f"{trough.delta_pct:+.2%}"
    print(f"{metric:<11} trough  baseline={trough.baseline:>15,.0f}  "
          f"stressed={trough.stressed:>15,.0f}  delta={trough.delta_absolute:>15,.0f}  {pct}")

assert result.baseline.invariants_hold and result.stressed.invariants_hold
assert result.stressed.horizon_months == 12

scenario: recession | horizon: 12
result hash: debc86fd7d621361 ...
dimension deltas: None -> deferred
revenue     trough  baseline=      6,336,414  stressed=        938,298  delta=     -5,398,115  -85.19%
ebitda      trough  baseline=        580,869  stressed=       -461,141  delta=     -1,042,010  -179.39%
net_income  trough  baseline=        157,368  stressed=       -811,005  delta=       -968,373  -615.36%
min_cash    trough  baseline=     12,687,848  stressed=     13,418,205  delta=        730,356  +5.76%


## 2. KPI rows in frozen order, with an explicit reason when Δ % is undefined

Frozen §5 lists revenue, EBITDA, net income, min cash, runway and min DSCR at
trough *and* end-horizon. Δ % is `None` **with a reason** when the baseline is
zero or unavailable — a missing number is never silently a zero.

In [3]:
print(f"{'metric':<11}{'scope':<7}{'delta':>16}  pct_reason")
for impact in result.kpi_impacts:
    print(f"{impact.metric:<11}{impact.scope:<7}{impact.delta_absolute:>16,.2f}  "
          f"{impact.delta_pct_reason or '-'}")

assert [(i.metric, i.scope) for i in result.kpi_impacts] == [
    (m, s) for m in st.KPI_ORDER for s in ("trough", "end")
]

flat = st.run_stress(dataset, sc.validate_params({"name": "flat"}, origin=sc.ScenarioOrigin.FORM))
reasons = {i.delta_pct_reason for i in flat.kpi_impacts if i.baseline == 0.0}
assert reasons in (set(), {"zero_baseline"})
print("flat run: delta_pct_reason values for zero baselines ->", sorted(reasons))

metric     scope             delta  pct_reason
revenue    trough    -5,398,115.08  -
revenue    end       -5,398,115.08  -
ebitda     trough    -1,042,010.33  -
ebitda     end       -1,042,010.33  -
net_income trough      -968,372.80  -
net_income end         -984,259.05  -
min_cash   trough       730,356.44  -
min_cash   end        1,211,128.70  -
runway     trough             0.00  -
runway     end                0.00  -
min_dscr   trough            -2.17  -
min_dscr   end               -3.32  -
flat run: delta_pct_reason values for zero baselines -> []


## 3. Breach policy (frozen §5 defaults, configurable per call)

Breach output is deterministic status/records only — no severity labels. The
current-ratio proxy rule reports `unevaluable`: the twin projects no
`current_assets`/`current_liabilities` levels and the frozen spec defines no
proxy formula, so Phase 8 does not invent one.

In [4]:
print(f"{'rule':<32}{'status':<14}{'threshold':>10}  periods")
for breach in result.breaches:
    threshold = "-" if breach.threshold is None else f"{breach.threshold:g}"
    print(f"{breach.rule:<32}{breach.status:<14}{threshold:>10}  {breach.periods}")

assert [b.rule for b in result.breaches] == list(st.BREACH_ORDER)
proxy = next(b for b in result.breaches if b.rule == "current_ratio_proxy_below_1")
assert proxy.status == "unevaluable" and proxy.observed is None

buffer = st.resolve_min_cash_buffer(result.stressed)
cash = result.stressed.trajectory("cash")
cash_rule = next(b for b in result.breaches if b.rule == "cash_below_buffer")
assert cash_rule.periods == [t + 1 for t, value in enumerate(cash) if value < buffer]
print("buffer B:", f"{buffer:,.2f}", "| trough cash:", f"{min(cash):,.2f}")

strict = st.run_stress(dataset, recession, policy=st.BreachPolicy(dscr=5.0))
print("custom DSCR threshold ->", next(
    b.threshold for b in strict.breaches if b.rule == "dscr_below_1_2"
))

rule                            status         threshold  periods
runway_below_6_months           not_breached           6  []
dscr_below_1_2                  breached             1.2  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
interest_coverage_below_1_5     breached             1.5  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
cash_below_buffer               not_breached  1.30584e+06  []
funding_gap_occurred            not_breached           -  []
current_ratio_proxy_below_1     unevaluable            1  []
buffer B: 1,305,844.75 | trough cash: 13,418,204.76
custom DSCR threshold -> 5.0


## 4. EBITDA waterfall and the below-EBITDA bridge

Frozen §5 order, computed month-by-month and then summed:
revenue effect `(Rev_s − Rev_b) · GM_b` → input-cost effect → opex effect →
other/one-off; then interest and tax carry EBITDA to net income. Closure is
asserted against the frozen 0.5% tolerance, and the revenue effect is
re-derived here from the emitted months to show the engine is not trust-me.

In [5]:
print("waterfall (ΔEBITDA contribution):")
for component in result.waterfall:
    print(f"  {component.component:<20}{component.delta_ebitda:>18,.2f}")
print("below-EBITDA bridge (ΔNI contribution):")
for component in result.below_bridge:
    print(f"  {component.component:<20}{component.delta_ni:>18,.2f}")

delta_ebitda = sum(m.ebitda for m in result.stressed.months) - sum(
    m.ebitda for m in result.baseline.months
)
parts = sum(c.delta_ebitda for c in result.waterfall)
delta_ni = sum(m.net_income for m in result.stressed.months) - sum(
    m.net_income for m in result.baseline.months
)
bridge = sum(c.delta_ni or 0.0 for c in result.below_bridge)
print("closure: dEBITDA", f"{delta_ebitda:,.2f}", "vs parts", f"{parts:,.2f}",
      "| dNI", f"{delta_ni:,.2f}", "vs EBITDA+bridge", f"{delta_ebitda + bridge:,.2f}")

# Hand re-derivation of the revenue leg from the emitted months.
hand = 0.0
for base, stress in zip(result.baseline.months, result.stressed.months, strict=True):
    gm = (base.revenue - base.cogs) / base.revenue if base.revenue else 0.0
    hand += (stress.revenue - base.revenue) * gm
engine = result.waterfall[0].delta_ebitda
relative = abs(hand - engine) / abs(engine)
print("revenue effect: engine", f"{engine:,.6f}", "hand", f"{hand:,.6f}",
      f"| relative residual {relative:.3e}")

assert abs(delta_ebitda - parts) <= max(0.005 * abs(delta_ebitda), 1e-9)
assert abs(delta_ni - (delta_ebitda + bridge)) <= max(0.005 * abs(delta_ni), 1e-9)
assert relative < 1e-12

waterfall (ΔEBITDA contribution):
  revenue_effect          -13,385,750.13
  input_cost_effect                 0.00
  opex_effect               4,737,323.06
  other_one_off                    -0.00
below-EBITDA bridge (ΔNI contribution):
  interest_effect                   0.00
  tax_effect                  661,929.29
closure: dEBITDA -8,648,427.08 vs parts -8,648,427.08 | dNI -7,986,497.79 vs EBITDA+bridge -7,986,497.79
revenue effect: engine -13,385,750.134708 hand -13,385,750.134708 | relative residual 0.000e+00


## 5. Reproducibility (frozen §6.2 / R6)

Identical inputs must give float-identical outputs. The `result_hash` is a
SHA-256 over the canonical JSON of the whole comparison, so equality of the
hash is equality of every number in the result.

In [6]:
again = st.run_stress(dataset, recession)
print("first :", result.result_hash)
print("second:", again.result_hash)
print("equal :", result == again)
assert again.result_hash == result.result_hash
assert again == result

other = st.run_stress(
    dataset, sc.validate_params({"name": "mild"}, origin=sc.ScenarioOrigin.FORM)
)
print("a different scenario gives a different hash:", other.result_hash != result.result_hash)
assert other.result_hash != result.result_hash

first : debc86fd7d621361a183ab3b7acaff70ef8a589ece5f9664a3e50e2409ae4c94
second: debc86fd7d621361a183ab3b7acaff70ef8a589ece5f9664a3e50e2409ae4c94
equal : True
a different scenario gives a different hash: True


## 6. Single-factor sweeps (frozen §6.3), and the finding D-6-3

A sweep varies **one** shock field over its frozen §3 bounds while every other
shock stays at zero — one-factor-at-a-time from the same zero-override baseline
`run_stress` uses — and includes the zero-shock baseline point. Frozen §6.3
claims trough cash and EBITDA move monotonically; the twin does **not** honour
that for trough cash on a contraction, because a shrinking business releases
working capital (`OCF = NI + DA − ΔNWC`). The engine reports the curve and
never asserts a direction, and the contradiction is pinned here rather than
smoothed over.

One further honest reading of the table above: on this seed the **upside is
flat** — every positive point equals the baseline. The twin caps revenue at
`revenue_capacity` (the trailing 12-month maximum, frozen §1 assumption), and
seed 1001 runs *at that ceiling in every baseline month*: a positive revenue
shock only raises `revenue_demand`, while `revenue = min(demand, capacity)` is
unchanged. That is a property of the frozen twin (§2), not a sweep bug — the
cell below prints the baseline months sitting on the ceiling and pins the flat
upside so neither reading can be mistaken for the other.

In [7]:
sweep = sn.sweep_single_factor(dataset, recession, "revenue_change_pct")
print("field:", sweep.field, "| reference:", sweep.reference, "| points:", len(sweep.points))

def trough(point, metric):
    return next(
        i.stressed for i in point.kpi_impacts
        if i.metric == metric and i.scope == "trough"
    )

print(f"{'revenue_change_pct':>19}{'trough cash':>16}{'trough EBITDA':>17}  breached")
for point in sweep.points:
    marker = "  <- baseline" if point.is_baseline_point else ""
    print(f"{point.value:>19.2f}{trough(point, 'min_cash'):>16,.0f}"
          f"{trough(point, 'ebitda'):>17,.0f}  {','.join(point.breaches) or '-'}{marker}")

lower, upper = sc.FIELD_BOUNDS["revenue_change_pct"]
values = [p.value for p in sweep.points]
assert values[0] == lower and values[-1] == upper
assert sum(1 for p in sweep.points if p.is_baseline_point) == 1
assert values == sorted(values)
print("breach onset by rule:", sweep.breach_onset)

# Capacity ceiling: seed 1001's baseline runs at revenue_capacity every month,
# so positive shocks raise demand only and the upside curve is flat.
capped = [m.period_index for m in result.baseline.months if m.revenue_capped]
print("baseline months at the capacity ceiling:", capped)
assert len(capped) == len(result.baseline.months)
by_value = {p.value: p for p in sweep.points}
baseline_ebitda = trough(by_value[0.0], "ebitda")
upside = [trough(by_value[v], "ebitda") for v in (12.5, 25.0, 37.5, 50.0)]
assert all(e == baseline_ebitda for e in upside)
print("upside points identical to baseline:", upside == [baseline_ebitda] * 4)

field: revenue_change_pct | reference: zero_shock_baseline | points: 9
 revenue_change_pct     trough cash    trough EBITDA  breached
             -50.00      11,138,876         -641,952  dscr_below_1_2,interest_coverage_below_1_5
             -37.50      11,917,054         -637,739  dscr_below_1_2,interest_coverage_below_1_5
             -25.00      13,134,889         -601,928  dscr_below_1_2,interest_coverage_below_1_5
             -12.50      14,499,661         -385,791  dscr_below_1_2,interest_coverage_below_1_5
               0.00      12,687,848          580,869  dscr_below_1_2  <- baseline
              12.50      12,687,848          580,869  dscr_below_1_2
              25.00      12,687,848          580,869  dscr_below_1_2
              37.50      12,687,848          580,869  dscr_below_1_2
              50.00      12,687,848          580,869  dscr_below_1_2
breach onset by rule: {'dscr_below_1_2': -50.0, 'interest_coverage_below_1_5': -50.0}
baseline months at the capacity ce

### D-6-3 pinned: a deeper contraction can *raise* trough cash

Seed 1002, the same fixture the Phase 6 finding was recorded on. The sweep is
re-run on that company below and the contradiction is asserted explicitly, so
it can never be quietly "fixed" into a monotonicity claim.

In [8]:
dataset_1002 = generate_company(SyntheticCompanyConfig(seed=1002, periods=24))
sweep_1002 = sn.sweep_single_factor(
    dataset_1002,
    sc.validate_params({"name": "d63"}, origin=sc.ScenarioOrigin.FORM),
    "revenue_change_pct",
)
by_value = {p.value: p for p in sweep_1002.points}
cash_at = lambda v: trough(by_value[v], "min_cash")  # noqa: E731
ebitda_at = lambda v: trough(by_value[v], "ebitda")  # noqa: E731

print(f"trough cash  at -37.5% revenue: {cash_at(-37.5):,.0f}")
print(f"trough cash  at    0.0% revenue: {cash_at(0.0):,.0f}")
print(f"trough EBITDA at -37.5% revenue: {ebitda_at(-37.5):,.0f}")
print(f"trough EBITDA at    0.0% revenue: {ebitda_at(0.0):,.0f}")

assert cash_at(-37.5) > cash_at(0.0)      # WC release lifts trough cash
assert ebitda_at(-37.5) < ebitda_at(0.0)  # while EBITDA falls
print("D-6-3: trough-cash monotonicity contradiction reproduced.")

trough cash  at -37.5% revenue: 19,196,824
trough cash  at    0.0% revenue: 14,107,653
trough EBITDA at -37.5% revenue: 4,321
trough EBITDA at    0.0% revenue: 1,189,520
D-6-3: trough-cash monotonicity contradiction reproduced.


## 7. What this notebook does **not** claim

- **Trough-cash monotonicity** (frozen §6.3) is contradicted by the twin on a
  contraction and is recorded, not resolved; amending a frozen invariant needs
  change control.
- **Current-ratio proxy breach** is `unevaluable`: the twin projects no
  balance-sheet levels and the frozen spec defines no proxy formula, so no
  number is invented for it.
- **Dimension deltas** (frozen §5) are deferred: re-scoring the stressed
  end-state needs a full `CompanyDataset` the twin never projects.
- Custom breach thresholds are per-call only; nothing is persisted (Phase 10).
- Sweeps are one-factor-at-a-time from the zero-shock base, so they measure
  single-factor sensitivity, not interaction effects.
- Outputs are an assumption-based intervention on a model under synthetic
  fixtures — not a forecast, not causal truth and not financial advice.